In [1]:

import baostock as bs
import pandas as pd
from datetime import datetime

def get_target_stocks(target_date=None):
    # 如果没有传入日期，默认使用今天（注意：如果是周末或节假日，需自行传入最近的交易日）
    if target_date is None:
        target_date = datetime.today().strftime('%Y-%m-%d')

    print(f"正在登录 Baostock 并获取 {target_date} 的数据...")
    bs.login()

    # 1. 获取当天所有的股票代码
    rs_all = bs.query_all_stock(day=target_date)
    stock_df = rs_all.get_data()

    if stock_df.empty:
        print("未获取到股票列表，请检查日期是否为交易日。")
        bs.logout()
        return None

    data_list = []
    total_stocks = len(stock_df)

    print(f"共获取到 {total_stocks} 只股票，开始逐个拉取数据（此过程可能需要几分钟到十几分钟，请耐心等待）...")

    # 2. 遍历所有股票，获取所需的 K线和估值数据
    for index, row in stock_df.iterrows():
        code = row['code']
        # 排除北交所或无效代码，仅保留沪深A股 (sh.6, sz.0, sz.3 开头)
        if not (code.startswith('sh.6') or code.startswith('sz.0') or code.startswith('sz.3')):
            continue

        # 获取数据：收盘价，交易状态，是否ST，滚动市盈率，市净率，成交量(股)，换手率(%)
        rs_data = bs.query_history_k_data_plus(
            code,
            "code,close,tradestatus,isST,peTTM,pbMRQ,volume,turn",
            start_date=target_date, end_date=target_date,
            frequency="d", adjustflag="3"
        )

        if rs_data.error_code == '0' and len(rs_data.data) > 0:
            row_data = rs_data.get_data().iloc[0]

            # 数据清洗与类型转换
            try:
                close = float(row_data['close'])
                tradestatus = str(row_data['tradestatus'])
                isST = str(row_data['isST'])
                pe = float(row_data['peTTM']) if row_data['peTTM'] else -1.0
                pb = float(row_data['pbMRQ']) if row_data['pbMRQ'] else -1.0
                volume = float(row_data['volume']) if row_data['volume'] else 0.0
                turn = float(row_data['turn']) if row_data['turn'] else 0.0
            except ValueError:
                continue

            # 3. 核心条件过滤
            # 非停牌 (tradestatus == '1')
            # 非ST (isST == '0')
            # PE > 0 且 PB > 0
            if tradestatus == '1' and isST == '0' and pe > 0 and pb > 0:
                # 4. 计算流通市值 (换手率单位是%，所以需要除以100)
                if turn > 0:
                    # 流通股本 = volume / (turn / 100)
                    # 流通市值 = close * 流通股本
                    market_cap = close * (volume / (turn / 100))

                    data_list.append({
                        'code': code,
                        'close': close,
                        'pe': pe,
                        'pb': pb,
                        'market_cap': market_cap
                    })

    bs.logout()
    print("数据抓取完成，正在进行排序过滤...")

    # 将结果转换为 DataFrame
    result_df = pd.DataFrame(data_list)

    if result_df.empty:
        print("没有符合条件的股票。")
        return result_df

    # 5. 按市值从小到大排序，并截取前 50 只
    top_50_smallest_cap = result_df.sort_values(by='market_cap', ascending=True).head(50)

    # 6. 在这 50 只股票的基础上，按价格由低到高排序
    final_result = top_50_smallest_cap.sort_values(by='close', ascending=True)

    # 重置索引
    final_result = final_result.reset_index(drop=True)

    return final_result

# 运行代码（你可以把 '2023-10-20' 替换成你想查询的最近一个有效交易日）
# 例如: df = get_target_stocks('2023-10-20')
# 如果不填日期，默认使用今天的日期。若今天是周末，请务必手动传入周五的日期。
if __name__ == '__main__':
    # 这里建议填入最近的一个交易日，例如 '2023-10-20'
    # df = get_target_stocks('2023-10-20')
    df = get_target_stocks('2026-04-01')

    if df is not None and not df.empty:
        print("\n=== 最终筛选结果（市值最小的前50只，并按价格由低到高排序） ===")
        # 打印时调整市值的显示格式（转换为“亿元”单位方便阅读）
        df['market_cap_亿元'] = df['market_cap'] / 100000000
        pd.set_option('display.max_rows', 50)
        print(df[['code', 'close', 'pe', 'pb', 'market_cap_亿元']])

正在登录 Baostock 并获取 2026-04-01 的数据...
login success!



KeyboardInterrupt

